## Technical Assessment


## Question 1

### Importing required libraries


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

data_path = Path("..") / "data" / "Housing.csv"
df = pd.read_csv(data_path)
df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


### Inspecting the structure

In [2]:
print("Shape:", df.shape)
print()
df.info()

print()
print("Missing values per column:")
print(df.isna().sum())

print()
df.describe(include="all")

Shape: (545, 13)

<class 'pandas.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   price             545 non-null    int64
 1   area              545 non-null    int64
 2   bedrooms          545 non-null    int64
 3   bathrooms         545 non-null    int64
 4   stories           545 non-null    int64
 5   mainroad          545 non-null    str  
 6   guestroom         545 non-null    str  
 7   basement          545 non-null    str  
 8   hotwaterheating   545 non-null    str  
 9   airconditioning   545 non-null    str  
 10  parking           545 non-null    int64
 11  prefarea          545 non-null    str  
 12  furnishingstatus  545 non-null    str  
dtypes: int64(6), str(7)
memory usage: 55.5 KB

Missing values per column:
price               0
area                0
bedrooms            0
bathrooms           0
stories             0
mainroad            0
guestr

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
count,5.450000e+02,545.000000,545.000000,545.000000,545.000000,545,545,545,545,545,545.000000,545,545
unique,NaN,NaN,NaN,NaN,NaN,2,2,2,2,2,NaN,2,3
top,NaN,NaN,NaN,NaN,NaN,yes,no,no,no,no,NaN,no,semi-furnished
freq,NaN,NaN,NaN,NaN,NaN,468,448,354,520,373,NaN,417,227
mean,4.766729e+06,5150.541284,2.965138,1.286239,1.805505,NaN,NaN,NaN,NaN,NaN,0.693578,NaN,NaN
std,1.870440e+06,2170.141023,0.738064,0.502470,0.867492,NaN,NaN,NaN,NaN,NaN,0.861586,NaN,NaN
min,1.750000e+06,1650.000000,1.000000,1.000000,1.000000,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN
25%,3.430000e+06,3600.000000,2.000000,1.000000,1.000000,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN
50%,4.340000e+06,4600.000000,3.000000,1.000000,2.000000,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN
75%,5.740000e+06,6360.000000,3.000000,2.000000,2.000000,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,NaN


### Inspecting the types 

In [3]:
df.dtypes

price               int64
area                int64
bedrooms            int64
bathrooms           int64
stories             int64
mainroad              str
guestroom             str
basement              str
hotwaterheating       str
airconditioning       str
parking             int64
prefarea              str
furnishingstatus      str
dtype: object

### Inspecting the anamolies

In [4]:
# Nulls and duplicates
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print("Missing values (non-zero):")
print(missing if not missing.empty else "None")
print()
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

# Potential datatype misconfigurations
object_cols = df.select_dtypes(include=["object"]).columns
numeric_cols = df.select_dtypes(include=["number"]).columns

non_numeric_report = {}
for col in df.columns:
    if col in object_cols:
        coerced = pd.to_numeric(df[col], errors="coerce")
        if coerced.notna().any():
            non_numeric_report[col] = int(coerced.isna().sum())

print("\nObject columns that look numeric (coercion failures count):")
print(non_numeric_report if non_numeric_report else "None")

# Whitespace and casing anomalies in object columns
print("\nObject column unique values (top 10) and counts:")
for col in object_cols:
    value_counts = df[col].value_counts(dropna=False).head(10)
    print(f"\n{col}:")
    print(value_counts)
    # Check for leading/trailing whitespace
    stripped = df[col].astype(str).str.strip()
    if not (stripped == df[col].astype(str)).all():
        print("  Warning: leading/trailing whitespace detected")

# Basic numeric sanity checks
print("\nNumeric columns with negative values:")
negatives = {col: int((df[col] < 0).sum()) for col in numeric_cols}
negatives = {col: cnt for col, cnt in negatives.items() if cnt > 0}
print(negatives if negatives else "None")

print("\nNumeric columns with zero values:")
zeros = {col: int((df[col] == 0).sum()) for col in numeric_cols}
zeros = {col: cnt for col, cnt in zeros.items() if cnt > 0}
print(zeros if zeros else "None")

Missing values (non-zero):
None

Duplicate rows: 0

Object columns that look numeric (coercion failures count):
None

Object column unique values (top 10) and counts:

mainroad:
mainroad
yes    468
no      77
Name: count, dtype: int64

guestroom:
guestroom
no     448
yes     97
Name: count, dtype: int64

basement:
basement
no     354
yes    191
Name: count, dtype: int64

hotwaterheating:
hotwaterheating
no     520
yes     25
Name: count, dtype: int64

airconditioning:
airconditioning
no     373
yes    172
Name: count, dtype: int64

prefarea:
prefarea
no     417
yes    128
Name: count, dtype: int64

furnishingstatus:
furnishingstatus
semi-furnished    227
unfurnished       178
furnished         140
Name: count, dtype: int64

Numeric columns with negative values:
None

Numeric columns with zero values:
{'parking': 299}


/var/folders/k8/35vdm9jn2t53z_ftr75fvb8w0000gn/T/ipykernel_40848/3642801836.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df.select_dtypes(include=["object"]).columns


## Question 2

### Data Preprocessing

Missing values before cleaning:
price               0
area                0
bedrooms            0
bathrooms           0
stories             0
mainroad            0
guestroom           0
basement            0
hotwaterheating     0
airconditioning     0
parking             0
prefarea            0
furnishingstatus    0
dtype: int64

Shape after dropping duplicates: (545, 13)


NameError: name 'cat_cols' is not defined